In [1]:
# 라이브러리 불러오기
import FinanceDataReader as fdr
import os

In [2]:
# 데이터 분석 패키지
import numpy as np
import pandas as pd

In [3]:
# 시각화 패키지
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [4]:
# 날짜는 파이썬 표준 라이브러리 datetime 사용
import datetime
from dateutil.relativedelta import relativedelta

In [5]:
# DB연결 패키지
import cx_Oracle
from sqlalchemy import create_engine, text

In [6]:
# 오늘 날짜 구하기
today = datetime.datetime.today()
today = today.strftime('%Y-%m-%d')
today

'2025-02-27'

In [7]:
# 기간별 날짜 반환 함수


def calculate_start_date(end_date, *, days_ago=None, months_ago=None):
    """
    기간별 날짜 반환 함수
    :param end_date: 기준이 되는 끝 날짜 (YYYY-MM-DD 형식)
    :param days_ago: 기준 날짜에서 몇 일 전인지 (정수)
    :param months_ago: 기준 날짜에서 몇 달 전인지 (정수)
    :return: 계산된 시작 날짜 (YYYY-MM-DD 형식)
    """
    end_date_obj = datetime.datetime.strptime(end_date, '%Y-%m-%d')
    
    # 둘 다 제공된 경우 예외 발생
    if days_ago is not None and months_ago is not None:
        raise ValueError("days_ago와 months_ago는 동시에 사용할 수 없습니다.")
    
    # 둘 다 제공되지 않은 경우 예외 발생
    if days_ago is None and months_ago is None:
        raise ValueError("days_ago 또는 months_ago 중 하나는 반드시 지정해야 합니다.")
    
    # 날짜 계산
    if days_ago is not None:
        start_date = end_date_obj - relativedelta(days=days_ago)
    elif months_ago is not None:
        start_date = end_date_obj - relativedelta(months=months_ago)
    
    return start_date.strftime('%Y-%m-%d')

In [8]:
# 주식 데이터를 기간별로 가져오는 함수
def get_stock_data(stock_code, end_date=today, *, days_ago=None, months_ago=1):
    try:
        # 시작 날짜 계산
        start_date = calculate_start_date(end_date, days_ago=days_ago, months_ago=months_ago)
        
        # 데이터 프레임 가져오기
        df_stock = fdr.DataReader(stock_code, start_date, end_date).reset_index()
        
        return df_stock

    except ValueError as ve:
        print(f"ValueError: {ve}")
        return None  # 또는 적절한 에러 처리 방법
    except KeyError as ke:
        print(f"KeyError: {ke}. 주식 코드 딕셔너리에 필요한 키가 없습니다.")
        return None  # 또는 적절한 에러 처리 방법
    except Exception as e:
        print(f"예기치 않은 오류 발생: {e}")
        return None  # 또는 적절한 에러 처리 방법

In [9]:
# Oracle 데이터베이스 연결 정보
username = 'c##PROJECT'       # 사용자명
password = 'k5002'        # 비밀번호
host = 'localhost'                # 호스트 (예: localhost)
port = '1521'                     # 포트 (기본값: 1521)
sid = 'xe'                  # SID

# DSN 생성 (SID를 사용)
dsn = cx_Oracle.makedsn(host, port, sid=sid)
connection_string = f'oracle+cx_oracle://{username}:{password}@{dsn}'

# SQLAlchemy 엔진 생성
engine = create_engine(connection_string)

In [10]:
# 함수에 사용할 종목 코드 리스트트

stk_codes_list = ['028300']

In [11]:
# SQL 쿼리 작성
query = """
  SELECT 
      s.STK_CODE,
      TO_CHAR(n.PUBLISHED_DATE, 'YYYY-MM-DD') AS PUBLISHED_DATE,
      SUM(n.NEWS_POS_LABEL) / COUNT(n.NEWS_ID) * 100 AS POS_SCORE
  FROM 
      NEWS n
  JOIN 
      STOCKS s ON n.STK_ID = s.STK_ID
  WHERE 
      s.STK_CODE = :stk_code
  GROUP BY 
      s.STK_CODE, TO_CHAR(n.PUBLISHED_DATE, 'YYYY-MM-DD')
  ORDER BY 
      PUBLISHED_DATE
"""

In [12]:
# 한국거래소(KRX) 상장 종목 리스트 가져오기
stk_list = fdr.StockListing('KRX')

In [13]:
# 긍정점수와 캔들차트에 쓰일 부분들을 합치는 함수
def candle_with_pos(stk_code_value, *, months_ago=1):
    try:
        # 종목 코드로 해당 종목 이름 찾기
        stock_name = stk_list[stk_list['Code'] == stk_code_value]['Name'].values[0]
        
        with engine.connect() as connection:
            db_df = pd.read_sql(text(query), connection, params={"stk_code": stk_code_value})
            db_df.columns = ['Stk_code', 'Date', 'Pos_score']
            db_df['Date'] = pd.to_datetime(db_df['Date'])

        print("데이터를 성공적으로 가져왔습니다!")
        
        # stk_code 열 제거
        db_df = db_df.drop(columns=['Stk_code'])

        # 주가 데이터 가져오기
        stock_data_df = get_stock_data(stk_code_value, months_ago=months_ago)

        # 주가 데이터와 병합
        merged_df = pd.merge(stock_data_df, db_df, on='Date', how='left')

        
        # # 긍정 점수 부분에 NaN 값이 있는 경우 0~100 사이의 임의의 숫자로 채우기
        # # 긍정 점수 계산 데이터를 가져왔을 때 nan값이면 임의의 샘플을 추가한다.
        # # 이 부분 없애도 
        # na_count = merged_df['Pos_score'].isna().sum()
        # if na_count > 0:
        #     random_scores = np.random.randint(0, 101, size=na_count)
        #     merged_df.loc[merged_df['Pos_score'].isna(), 'Pos_score'] = random_scores

        
        # 열 이름 변경
        merged_df.columns = ['날짜', '시가', '고가', '저가', '종가', '거래량', '전일비', '긍정점수']

        # 서브플롯 생성: 2행 1열의 그리드
        fig = make_subplots(
            rows=2, 
            cols=1, 
            specs=[[{"secondary_y": True}], [{}]],  # 두 번째 행은 기본 y축만 사용
            row_heights=[0.75, 0.25]  # 첫 번째 행은 75%, 두 번째 행은 25% 높이
        )

        # 캔들차트 추가
        fig.add_trace(go.Candlestick(x=merged_df['날짜'],
                                      open=merged_df['시가'],
                                      high=merged_df['고가'],
                                      low=merged_df['저가'],
                                      close=merged_df['종가'],
                                      name='캔들차트'),
                      row=1, col=1)  # 첫 번째 행, 첫 번째 열에 추가

        # 긍정 점수의 꺾은선 그래프 추가 (오른쪽 y축)
        fig.add_trace(go.Scatter(x=merged_df['날짜'], 
                                 y=merged_df['긍정점수'],  # 긍정 점수 데이터 사용
                                 name='긍정 점수',
                                 line=dict(color='skyblue')),
                      secondary_y=True, row=1, col=1)  # 첫 번째 행, 첫 번째 열에 추가

        # 거래량 바차트 추가 (두 번째 행)
        fig.add_trace(go.Bar(x=merged_df['날짜'], 
                             y=merged_df['거래량'], 
                             name='거래량',
                             marker=dict(color='lightgray')),
                      row=2, col=1)  # 두 번째 행, 첫 번째 열에 추가

        # x축 범위 슬라이더 비활성화
        fig.update_xaxes(rangeslider_visible=False)

        # y축 레이블 설정 및 긍정 점수 y축 범위 설정
        fig.update_layout(
            title=f'{stock_name}: {months_ago}개월 긍점점수와 시세 변동 추이',
            xaxis_title='날짜',
            yaxis_title='가격',
            yaxis2_title='긍정 점수',
            height=800,  # 차트 전체 높이 조정
        )

        # 긍정 점수 y축 범위 설정
        fig.update_yaxes(range=[0, 100], secondary_y=True)

        # 차트 저장할 경로 설정
        base_dir = 'chart_html'
        file_name = f'{stk_code_value}_candle_pos_{months_ago}m.html'

        # HTML 파일로 저장
        html_file_path = os.path.join(base_dir, file_name)
        fig.write_html(html_file_path)
        
        

    except Exception as e:
        print(f"오류 발생: {e}")
        return None


In [14]:
# 수익률을 계산하고 캔들차트를 그리는 함수
def candle_with_return(stk_code_value, *, months_ago=1):
    try:
        # 종목 코드로 해당 종목 이름 찾기
        stock_name = stk_list[stk_list['Code'] == stk_code_value]['Name'].values[0]
        
        # 주가 데이터 가져오기
        stock_data_df = get_stock_data(stk_code_value, months_ago=months_ago)

        # 수익률 계산
        first_close = stock_data_df['Close'].iloc[0]
        stock_data_df['Return'] = ((stock_data_df['Close'] - first_close) / first_close) * 100

        print(f"{stock_name} 데이터 가져오기 완료!")
        
        # 열 이름 변경
        stock_data_df.columns = ['날짜', '시가', '고가', '저가', '종가', '거래량', '전일비', '수익률']

        # 서브플롯 생성: 2행 1열의 그리드
        fig = make_subplots(
            rows=2, 
            cols=1, 
            specs=[[{"secondary_y": True}], [{}]],  # 두 번째 행은 기본 y축만 사용
            row_heights=[0.75, 0.25]  # 첫 번째 행은 75%, 두 번째 행은 25% 높이
        )

        # 캔들차트 추가
        fig.add_trace(go.Candlestick(x=stock_data_df['날짜'],
                                      open=stock_data_df['시가'],
                                      high=stock_data_df['고가'],
                                      low=stock_data_df['저가'],
                                      close=stock_data_df['종가'],
                                      name='캔들차트'),
                      row=1, col=1)  # 첫 번째 행, 첫 번째 열에 추가

        # 수익률의 꺾은선 그래프 추가 (오른쪽 y축)
        fig.add_trace(go.Scatter(x=stock_data_df['날짜'], 
                                 y=stock_data_df['수익률'],  # 수익률 데이터 사용
                                 name='수익률',
                                 line=dict(color='orange')),
                      secondary_y=True, row=1, col=1)  # 첫 번째 행, 첫 번째 열에 추가

        # 거래량 바차트 추가 (두 번째 행)
        fig.add_trace(go.Bar(x=stock_data_df['날짜'], 
                             y=stock_data_df['거래량'], 
                             name='거래량',
                             marker=dict(color='lightgray')),
                      row=2, col=1)  # 두 번째 행, 첫 번째 열에 추가

        # x축 범위 슬라이더 비활성화
        fig.update_xaxes(rangeslider_visible=False)

        # y축 레이블 설정 및 수익률 y축 범위 설정
        fig.update_layout(
            title=f'{stock_name}: {months_ago}개월 수익률과 시세 변동 추이',
            xaxis_title='날짜',
            yaxis_title='가격',
            yaxis2_title='수익률',
            height=800,  # 차트 전체 높이 조정
        )

        # 수익률 y축 범위 설정
        fig.update_yaxes(range=[min(stock_data_df['수익률']), max(stock_data_df['수익률'])], secondary_y=True)

        # 차트 저장할 경로 설정
        base_dir = 'chart_html'
        file_name = f'{stk_code_value}_candle_return_{months_ago}m.html'

        # HTML 파일로 저장
        html_file_path = os.path.join(base_dir, file_name)
        fig.write_html(html_file_path)
        
        

    except Exception as e:
        print(f"오류 발생: {e}")
        return None

In [15]:
# 캔들차트와 일일 변동성
def candle_with_volatility(stk_code_value, *, months_ago=1):
    try:
        # 종목 코드로 해당 종목 이름 찾기
        stock_name = stk_list[stk_list['Code'] == stk_code_value]['Name'].values[0]
        
        # 주가 데이터 가져오기
        stock_data_df = get_stock_data(stk_code_value, months_ago=months_ago)

        # 고가와 저가의 차이를 이용한 변동성 계산 (백분율)
        stock_data_df['Volatility'] = ((stock_data_df['High'] - stock_data_df['Low']) / stock_data_df['Close']) * 100

        print(f"{stock_name} 데이터 가져오기 완료!")
        
        # 열 이름 변경
        stock_data_df.columns = ['날짜', '시가', '고가', '저가', '종가', '거래량', '전일비', '변동성']

        # 서브플롯 생성: 2행 1열의 그리드
        fig = make_subplots(
            rows=2, 
            cols=1, 
            specs=[[{"secondary_y": True}], [{}]],  # 두 번째 행은 기본 y축만 사용
            row_heights=[0.75, 0.25]  # 첫 번째 행은 75%, 두 번째 행은 25% 높이
        )

        # 캔들차트 추가
        fig.add_trace(go.Candlestick(x=stock_data_df['날짜'],
                                      open=stock_data_df['시가'],
                                      high=stock_data_df['고가'],
                                      low=stock_data_df['저가'],
                                      close=stock_data_df['종가'],
                                      name='캔들차트'),
                      row=1, col=1)  # 첫 번째 행, 첫 번째 열에 추가

        # 변동성의 꺾은선 그래프 추가 (오른쪽 y축)
        fig.add_trace(go.Scatter(x=stock_data_df['날짜'], 
                                 y=stock_data_df['변동성'],  # 변동성 데이터 사용
                                 name='일일 변동성 (%)',
                                 line=dict(color='orange')),
                      secondary_y=True, row=1, col=1)  # 첫 번째 행, 첫 번째 열에 추가

        # 거래량 바차트 추가 (두 번째 행)
        fig.add_trace(go.Bar(x=stock_data_df['날짜'], 
                             y=stock_data_df['거래량'], 
                             name='거래량',
                             marker=dict(color='lightgray')),
                      row=2, col=1)  # 두 번째 행, 첫 번째 열에 추가

        # x축 범위 슬라이더 비활성화
        fig.update_xaxes(rangeslider_visible=False)

        # y축 레이블 설정
        fig.update_layout(
            title=f'{stock_name}: {months_ago}개월 변동성과 시세 변동 추이',
            xaxis_title='날짜',
            yaxis_title='가격',
            yaxis2_title='일일 변동성 (%)',
            height=800,  # 차트 전체 높이 조정
        )

        # 차트 저장할 경로 설정
        base_dir = 'chart_html'
        file_name = f'{stk_code_value}_candle_volatility_{months_ago}m.html'

        # HTML 파일로 저장
        html_file_path = os.path.join(base_dir, file_name)
        fig.write_html(html_file_path)
        
        
    except Exception as e:
        print(f"오류 발생: {e}")
        return None

In [16]:
# 각 종목 코드에 대해 n개월의 캔들차트 생성
for stk_code in stk_codes_list:
    months = 1  # 여기에 n개월 n값 입력
    candle_with_pos(stk_code, months_ago=months)
    candle_with_return(stk_code, months_ago=months)


데이터를 성공적으로 가져왔습니다!
HLB 데이터 가져오기 완료!
